In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import logging
from datetime import datetime
import json

# Configuración global
TEST_SIZE = 0.2  # Proporción del conjunto de prueba
RANDOM_STATE = 42  # Semilla para reproducibilidad
MAX_MEMORY_GB = 4  # Límite de memoria para cargar archivos grandes

# Configuración de logging
logging.basicConfig(
    filename='train_test_split.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def check_memory_usage(file_path):
    """
    Verifica si un archivo puede ser cargado en memoria sin exceder el límite configurado.
    """
    file_size_bytes = os.path.getsize(file_path)
    file_size_gb = file_size_bytes / (1024 ** 3)  # Convertir a GB
    logging.info(f"Tamaño del archivo {os.path.basename(file_path)}: {file_size_gb:.2f} GB")
    if file_size_gb > MAX_MEMORY_GB:
        logging.warning(f"El archivo {os.path.basename(file_path)} excede el límite de memoria.")
        return False
    return True

def validate_data_structure(df, target_col='nivel_triage'):
    """
    Valida la estructura del DataFrame y la distribución de clases.
    """
    # Verificar que la columna objetivo exista
    if target_col not in df.columns:
        raise ValueError(f"La columna objetivo '{target_col}' no existe en el DataFrame.")
    
    # Verificar que la columna objetivo tenga al menos dos clases
    unique_classes = df[target_col].unique()
    if len(unique_classes) < 2:
        raise ValueError(f"La columna objetivo '{target_col}' contiene solo una clase: {unique_classes}.")
    
    # Verificar el balance de clases
    class_distribution = df[target_col].value_counts()
    logging.info(f"Distribución de clases: {class_distribution.to_dict()}")
    
    # Verificar que todas las clases tengan suficientes muestras (solo advertencia)
    min_samples = 1000  # Umbral mínimo para advertencia
    if class_distribution.min() < min_samples:
        problematic_classes = class_distribution[class_distribution < min_samples].index.tolist()
        logging.warning(f"Advertencia: Las clases {problematic_classes} tienen menos de {min_samples} muestras.")
    
    return True

def calculate_class_weights(y):
    """
    Calcula pesos para cada clase basados en su frecuencia inversa.
    """
    class_counts = y.value_counts()
    total_samples = len(y)
    n_classes = len(class_counts)
    
    # Peso = total de muestras / (n_clases * frecuencia de la clase)
    class_weights = {
        cls: total_samples / (n_classes * count)
        for cls, count in class_counts.items()
    }
    
    return class_weights

def create_train_test_split(input_file, output_folder):
    """
    Crea conjuntos de entrenamiento y prueba manteniendo la distribución original.
    """
    try:
        logging.info(f"Procesando archivo: {os.path.basename(input_file)}")
        print(f"\n📂 Procesando archivo: {os.path.basename(input_file)}")
        
        # Verificar el uso de memoria antes de cargar el archivo
        if not check_memory_usage(input_file):
            logging.warning(f"El archivo {os.path.basename(input_file)} excede el límite de memoria. Saltando...")
            print(f"⚠️ El archivo {os.path.basename(input_file)} excede el límite de memoria. Saltando...")
            return False
        
        # Cargar el archivo
        df = pd.read_parquet(input_file)
        logging.info(f"Archivo cargado. Filas: {len(df)}, Columnas: {len(df.columns)}")
        print(f"📊 Archivo cargado. Filas: {len(df)}, Columnas: {len(df.columns)}")
        
        # Validar la estructura del DataFrame
        validate_data_structure(df)
        print("✅ Estructura de datos validada correctamente.")
        
        # Obtener distribución de clases inicial
        target_col = 'nivel_triage'
        class_distribution = df[target_col].value_counts().to_dict()
        
        # Calcular pesos de clase para usar en modelos
        class_weights = calculate_class_weights(df[target_col])
        
        # Dividir los datos en entrenamiento y prueba (estratificado)
        X = df.drop(columns=[target_col])
        y = df[target_col]
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=RANDOM_STATE
        )
        logging.info(f"División completada. Train: {len(X_train)}, Test: {len(X_test)}")
        print(f"✂️ División completada. Train: {len(X_train)}, Test: {len(X_test)}")
        
        # Verificar distribución de clases después del split
        train_distribution = y_train.value_counts().to_dict()
        test_distribution = y_test.value_counts().to_dict()
        
        # Guardar los conjuntos de datos
        os.makedirs(output_folder, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        
        # Guardar datos principales
        X_train.to_parquet(os.path.join(output_folder, f"{base_name}_X_train.parquet"))
        X_test.to_parquet(os.path.join(output_folder, f"{base_name}_X_test.parquet"))
        pd.DataFrame(y_train).to_parquet(os.path.join(output_folder, f"{base_name}_y_train.parquet"))
        pd.DataFrame(y_test).to_parquet(os.path.join(output_folder, f"{base_name}_y_test.parquet"))
        
        # Guardar metadatos útiles para entrenar modelos
        metadata = {
            "class_distribution": class_distribution,
            "train_distribution": train_distribution,
            "test_distribution": test_distribution,
            "class_weights": class_weights,
            "is_balanced": False,  # Indicar que no se aplicó balanceo
            "timestamp": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            "source_file": os.path.basename(input_file)
        }
        
        # Guardar metadatos en formato JSON
        with open(os.path.join(output_folder, f"{base_name}_metadata.json"), 'w') as f:
            json.dump(metadata, f, indent=4)
        
        logging.info(f"Archivos guardados en: {output_folder}")
        print(f"💾 Archivos guardados en: {output_folder}")
        print(f"📋 Distribución de clases en train: {train_distribution}")
        print(f"📋 Distribución de clases en test: {test_distribution}")
        print(f"⚖️ Pesos calculados por clase: {class_weights}")
        
        return True
    
    except Exception as e:
        logging.error(f"Error procesando {os.path.basename(input_file)}: {e}", exc_info=True)
        print(f"❌ Error procesando {os.path.basename(input_file)}: {e}")
        return False

def main():
    # Directorios de entrada y salida
    input_folder = r"C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\experimento_final_20250223_181000"
    output_folder = os.path.join(r"C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments", f"train_test_splits_no_smote_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    
    # Verificar que el directorio de entrada exista
    if not os.path.exists(input_folder):
        raise FileNotFoundError(f"El directorio de entrada no existe: {input_folder}")
    
    # Recorrer todos los subdirectorios y archivos .parquet
    success_count = 0
    total_files = 0
    
    print(f"🔍 Buscando archivos .parquet en: {input_folder}")
    for root, dirs, files in os.walk(input_folder):
        for file_name in files:
            if file_name.endswith(".parquet"):
                total_files += 1
                input_file = os.path.join(root, file_name)
                relative_path = os.path.relpath(root, input_folder)
                output_subfolder = os.path.join(output_folder, relative_path)
                
                print(f"\n📄 Procesando archivo {total_files}: {file_name}")
                if create_train_test_split(input_file, output_subfolder):
                    success_count += 1
    
    # Resumen final
    logging.info(f"Proceso completado. Archivos procesados correctamente: {success_count}/{total_files}")
    print(f"\n🎉 Proceso completado. Archivos procesados correctamente: {success_count}/{total_files}")
    print(f"\n📊 Generado sin balanceo artificial. Los archivos de metadatos contienen información de pesos por clase para usar en los modelos.")

if __name__ == "__main__":
    main()

🔍 Buscando archivos .parquet en: C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\experimento_final_20250223_181000

📄 Procesando archivo 1: MaxAbs_all_features.parquet

📂 Procesando archivo: MaxAbs_all_features.parquet
📊 Archivo cargado. Filas: 560486, Columnas: 713
✅ Estructura de datos validada correctamente.
✂️ División completada. Train: 448388, Test: 112098
💾 Archivos guardados en: C:\Users\Administrador\Documents\PythonScripts\Tesis\tesisaustral\outputs\experiments\train_test_splits_no_smote_20250512_225619\MaxAbs
📋 Distribución de clases en train: {5: 124361, 4: 100161, 3: 95370, 2: 86994, 1: 41502}
📋 Distribución de clases en test: {5: 31091, 4: 25040, 3: 23843, 2: 21748, 1: 10376}
⚖️ Pesos calculados por clase: {5: 0.721104906980933, 4: 0.8953378966621672, 3: 0.9403102010686754, 2: 1.0308546835629289, 1: 2.160784918462547}

📄 Procesando archivo 2: MaxAbs_Linear_selected.parquet

📂 Procesando archivo: MaxAbs_Linear_selected.parquet
📊 Archiv